# 09.12 · Agents and tool calling

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/09-nlp-genai/12-agents-tool-calling/lesson.ipynb)

Read the [README](README.md) first. We give the model three Python functions over ShopWave's data, let it decide which to call, watch it work, watch it fail, and then add guardrails in code.

> **About the outputs in this notebook.** They were produced with `llama3.2` (3B) via Ollama on a laptop. Yours will differ: a different model, a different run, even the same model on a different machine. Small local models make more mistakes than large hosted ones; that is part of what this lesson shows.

In [1]:
# --- course files: found locally, downloaded on Colab -------------------------
import sys, urllib.request
from pathlib import Path
REPO_RAW = "https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/"

def repo_file(rel):
    """Return a local path to a file in this repo, downloading it if needed."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists():
            return parent / rel
    dest = Path("course_files") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

In [2]:
sys.path.insert(0, str(repo_file("lib/course_llm.py").parent))
import course_llm
from course_llm import ask, ask_json, chat, info
info();

LLM provider: ollama · model: llama3.2


## 1. The tools

Ordinary Python functions. The model never runs code itself; it only **asks** for a tool by name, and our loop decides whether to run it.

In [3]:
sys.path.insert(0, str(repo_file("data/download.py").parent))
from download import load_retail
import re
sales = load_retail()                                   # ShopWave's order lines
docs_dir = repo_file("data/shopwave_docs/returns.md").parent

def order_status(invoice):
    """Look up one order by its 6-digit invoice number."""
    if not re.fullmatch(r"C?\d{6}", invoice.strip()):
        return "Error: an invoice number is 6 digits, for example 536365."
    lines = sales[sales["Invoice"] == invoice.strip()]
    if lines.empty:
        return f"No order found with invoice {invoice}."
    status = "cancelled" if invoice.startswith("C") else "delivered"
    total = (lines["Quantity"] * lines["Price"]).sum()
    return (f"Invoice {invoice}: placed {lines['InvoiceDate'].min():%d %b %Y}, {len(lines)} lines, "
            f"total £{total:,.2f}, shipped to {lines['Country'].iloc[0]}, status {status}.")

In [4]:
def refund_policy(_=""):
    """ShopWave's returns and refunds rules."""
    return ((docs_dir / "returns.md").read_text() + "\n" + (docs_dir / "refunds.md").read_text())

def product_search(query):
    """Find products whose name contains all the query words; show typical price."""
    words = query.lower().split()
    names = sales["Description"].fillna("").str.lower()
    hits = sales[names.apply(lambda n: all(w in n for w in words))]
    if hits.empty:
        return f"No products match '{query}'."
    top = hits.groupby("Description")["Price"].agg(["median", "count"]).nlargest(5, "count")
    return "; ".join(f"{d.title()} (£{r['median']:.2f})" for d, r in top.iterrows())

TOOL_FUNCTIONS = {"order_status": order_status, "refund_policy": refund_policy,
                  "product_search": product_search}

In [5]:
print(order_status("536365"))
print(product_search("hot water bottle"))
print(refund_policy()[:200], "…")

Invoice 536365: placed 01 Dec 2010, 7 lines, total £139.12, shipped to United Kingdom, status delivered.


Chocolate Hot Water Bottle (£4.95); Hot Water Bottle Keep Calm (£4.95); Scottie Dog Hot Water Bottle (£4.95); Hot Water Bottle I Am So Poorly (£4.95); Hot Water Bottle Tea And Sympathy (£4.25)
# Returns

You can return most items within **30 days of delivery** for a refund.

## Conditions

- Items must be unused and in their original packaging.
- **Personalised items** (engraved, monogramme …


## 2. The decision format

Each turn the model must reply with JSON: a short thought, a tool name (one of the three, or `final_answer`) and an argument. The JSON schema's `enum` means it cannot name a tool that does not exist.

In [6]:
import json
AGENT_SYSTEM = """You are ShopWave's support agent. You can use these tools:
- order_status(argument = 6-digit invoice number): date, total, country and status of an order
- refund_policy(argument = ""): ShopWave's returns and refunds rules
- product_search(argument = a few words of a product name): matching products and typical prices
Each turn, reply with JSON: a short thought, the tool to use, and its argument.
When you have enough information, use the tool "final_answer" with the answer to the customer as the argument.
Use only facts from tool results. Never invent order details or prices."""

STEP = {"type": "object", "properties": {
            "thought": {"type": "string"},
            "tool": {"type": "string", "enum": [*TOOL_FUNCTIONS, "final_answer"]},
            "argument": {"type": "string"}},
        "required": ["thought", "tool", "argument"], "additionalProperties": False}

## 3. The loop

Decide → call the tool → show the model the result → repeat, until it gives a final answer or runs out of steps. That loop *is* the agent.

In [7]:
def run_agent(question, max_steps=5, verbose=True):
    messages = [{"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        reply = chat(messages, system=AGENT_SYSTEM, max_tokens=400, temperature=0, schema=STEP)
        decision = json.loads(reply)
        if verbose:
            print(f"step {step}: {decision['tool']}({decision['argument']!r})  # {decision['thought']}")
        if decision["tool"] == "final_answer":
            return decision["argument"]
        result = TOOL_FUNCTIONS[decision["tool"]](decision["argument"])
        if verbose:
            print(f"        result: {result[:150]}")
        messages += [{"role": "assistant", "content": reply},
                     {"role": "user", "content": f"Tool result: {result}"}]
    return "Sorry, I could not finish this request. A colleague will reply by email."

In [8]:
QUESTIONS = [
    "What's the status of order 536365, and how much was it?",
    "Do you sell hot water bottles? Roughly how much are they?",
    "I bought a cushion three weeks ago and I don't like it. Can I get a refund?",
    "Order 536365: how much was it, and can I still return it today?",
]

In [9]:
for q in QUESTIONS:
    print(f"\nQ: {q}")
    print("ANSWER:", run_agent(q))


Q: What's the status of order 536365, and how much was it?


step 1: order_status('536365')  # Order 536365 needs to be checked for status and total.
        result: Invoice 536365: placed 01 Dec 2010, 7 lines, total £139.12, shipped to United Kingdom, status delivered.


step 2: refund_policy('')  # Order 536365 has been delivered, now need to check refund policy.
        result: # Returns

You can return most items within **30 days of delivery** for a refund.

## Conditions

- Items must be unused and in their original packagi


step 3: refund_policy('')  # Refund policy explained, now need to check if order 536365 qualifies for a refund.
        result: # Returns

You can return most items within **30 days of delivery** for a refund.

## Conditions

- Items must be unused and in their original packagi


step 4: final_answer('{"status": "delivered", "total": 139.12, "country": "United Kingdom"}')  # Order 536365 was delivered 01 Dec 2010, which is within the 30-day return window.
ANSWER: {"status": "delivered", "total": 139.12, "country": "United Kingdom"}

Q: Do you sell hot water bottles? Roughly how much are they?


...

step 3: refund_policy('')  # The order has been delivered, which means it's no longer eligible for a refund. However, we need to check the refund policy to confirm.
        result: # Returns

You can return most items within **30 days of delivery** for a refund.

## Conditions

- Items must be unused and in their original packagi


step 4: order_status('536365')  # Since the cushion was delivered 3 weeks ago, which is within the 30-day return window, and it's not a personalised item, food, or sale item, a refund is possible. We need to initiate the return process.
        result: Invoice 536365: placed 01 Dec 2010, 7 lines, total £139.12, shipped to United Kingdom, status delivered.


step 5: refund_policy('')  # The order has been delivered, and we need to initiate the return process. We'll need to contact the customer to arrange for the return and provide a prepaid returns label.
        result: # Returns

You can return most items within **30 days of delivery** for a refund.

## Conditions

- Items must be unused and in their original packagi
ANSWER: Sorry, I could not finish this request. A colleague will reply by email.

Q: Order 536365: how much was it, and can I still return it today?


step 1: order_status('536365')  # Order 536365 details needed
        result: Invoice 536365: placed 01 Dec 2010, 7 lines, total £139.12, shipped to United Kingdom, status delivered.


step 2: final_answer('{"status": "delivered", "return_policy": "No returns accepted after delivery"}')  # Order 536365 delivered, no returns possible
ANSWER: {"status": "delivered", "return_policy": "No returns accepted after delivery"}


Read the traces carefully. Typical things a small model does here (our run showed most of them):

- **Repeats the same call** several times, wasting steps and tokens.
- **Invents an argument**: asked about a cushion with no invoice number, it may call `order_status` with a made-up number, and then use **the example number from the tool's own error message** as if it were the customer's order. Tool results are part of the prompt; text in them steers the model.
- **Answers in JSON** instead of a sentence for the customer.
- **Reasons wrongly** from correct facts, for example deciding an order cannot be returned *because* it was delivered.
- **Runs out of steps** and falls back to the apology.

None of these is a crash. Each one would reach a customer.

## 4. Guardrails in code and in the prompt

Three changes:

1. A tool error message that **instructs** ("ask the customer for their invoice number") instead of giving an example to copy.
2. Rules in the system prompt: no repeated calls, ask for missing information, plain sentences.
3. A guardrail **in the loop itself**: a repeated call is not executed; the model is told to answer instead. Code enforces this whatever the model decides.

In [10]:
def order_status_checked(invoice):
    """Same lookup, but a bad number gets an instruction, not an example to copy."""
    if not re.fullmatch(r"C?\d{6}", invoice.strip()):
        return "No valid invoice number. Ask the customer for their 6-digit invoice number."
    return order_status(invoice)

GUARDED_TOOLS = {**TOOL_FUNCTIONS, "order_status": order_status_checked}
GUARDED_SYSTEM = AGENT_SYSTEM + """
Rules:
- Call each tool at most once with the same argument.
- If you need information the customer has not given (such as an invoice number), use final_answer to ask for it.
- Write the final answer as one or two plain sentences for the customer, never JSON."""

In [11]:
def run_guarded_agent(question, max_steps=5):
    messages, seen = [{"role": "user", "content": question}], set()
    for step in range(1, max_steps + 1):
        reply = chat(messages, system=GUARDED_SYSTEM, max_tokens=400, temperature=0, schema=STEP)
        d = json.loads(reply)
        print(f"step {step}: {d['tool']}({d['argument']!r})")
        if d["tool"] == "final_answer":
            return d["argument"]
        if (d["tool"], d["argument"]) in seen:                 # guardrail in code, not in the prompt
            result = "You already have this result. Answer the customer now with final_answer."
        else:
            seen.add((d["tool"], d["argument"]))
            result = GUARDED_TOOLS[d["tool"]](d["argument"])
        messages += [{"role": "assistant", "content": reply},
                     {"role": "user", "content": f"Tool result: {result}"}]
    return "Sorry, I could not finish this request. A colleague will reply by email."

In [12]:
for q in QUESTIONS:
    print(f"\nQ: {q}")
    print("ANSWER:", run_guarded_agent(q))


Q: What's the status of order 536365, and how much was it?


step 1: order_status('536365')


step 2: final_answer('Order 536365 was successfully delivered to the United Kingdom on 01 Dec 2010 for a total of £139.12.')
ANSWER: Order 536365 was successfully delivered to the United Kingdom on 01 Dec 2010 for a total of £139.12.

Q: Do you sell hot water bottles? Roughly how much are they?


step 1: product_search('hot water bottles')


step 2: product_search('hot water bottle')


...

step 1: refund_policy('')


step 2: order_status('1234567890')


step 3: final_answer('Can you please provide the 6-digit invoice number for the cushion you purchased three weeks ago?')
ANSWER: Can you please provide the 6-digit invoice number for the cushion you purchased three weeks ago?

Q: Order 536365: how much was it, and can I still return it today?


step 1: order_status('536365')


step 2: final_answer('You are not eligible for a return on this order as it has been shipped and delivered.')
ANSWER: You are not eligible for a return on this order as it has been shipped and delivered.


Compare the two runs question by question. In our run the guardrails fixed the JSON answers, stopped the invented invoice (the agent asked the customer for it instead) and blocked identical repeat calls. The hot water bottle question still took five steps, because the agent tried `"hot water bottles"` (which finds nothing: product names say "bottle") and then other spellings; that is a tool problem, fixed in exercise 1. What prompts cannot fix is **reasoning about dates**: the last question needs "today minus 1 December 2010 > 30 days", and a 3B model guesses. The fix is another tool that does the arithmetic in Python (exercise 2). The pattern of this whole module: let the model handle language and choose actions; let code handle facts, sums and permissions.

## 5. Agents in production

- **Native tool calling.** Real systems usually use the provider's own tool API instead of our JSON-in-text approach: you describe tools with a JSON schema, and the response contains structured tool-call blocks. See the [Anthropic tool use guide](https://docs.anthropic.com/en/docs/build-with-claude/tool-use) and [OpenAI function calling](https://platform.openai.com/docs/guides/function-calling). Ollama supports tools for some models too. The loop is the same as ours.
- **Least privilege.** Give an agent read-only tools by default. Anything that changes the world (refunds, emails, cancellations) needs a check in code or a human approval.
- **Budgets.** Cap steps, tokens and money per conversation.
- **Logs.** Keep every step's decision and result, as our traces do, so you can see why something went wrong.

## Quiz answers

1. A loop in which a model repeatedly decides on an action (a tool call), the code runs it, and the model sees the result, until the task is done.
2. Our code runs it, after checking the tool name and argument. The model only asks.
3. Text in tool results becomes part of the prompt, so it can steer the model (here: an example invoice number got copied).
4. A cap on steps (and tokens), so a confused agent cannot loop forever or run up a bill.
5. Anything with real-world effects that is hard to undo: issuing refunds, sending emails, cancelling orders, changing customer data.

**Next:** [09.P Project: review analyser and support chatbot](../project/)